In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
import time
import pandas as pd
import numpy as np
import random

from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score

## 1. Load Processed Graph Dataset

In [3]:
DATA_PROCESSED = Path("../data/processed")

def load_graph(name):
    path = DATA_PROCESSED / f"{name}_final.pt"
    if not path.exists():
        raise FileNotFoundError(f"{path} not found.")
    data = torch.load(path, weights_only=False)
    print(f"Loaded {name}")
    print(data)
    return data

dataset_name = "elliptic"
data = load_graph(dataset_name)

Loaded elliptic
Data(x=[203769, 165], edge_index=[2, 234355], y=[203769], timesteps=[203769], num_nodes=203769, train_mask=[203769], test_mask=[203769], val_mask=[203769])


In [4]:
def evaluate_predictions(y_true, y_pred, y_prob):
    return {
        "F1": f1_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "AUC": roc_auc_score(y_true, y_prob)
    }

### Basic GCN Layer (Undirected)


In [5]:
class BasicGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, x, edge_index, num_nodes):
        src, dst = edge_index

        # Make edges undirected
        src_all = torch.cat([src, dst])
        dst_all = torch.cat([dst, src])

        deg = torch.zeros(num_nodes, device=x.device)
        deg.scatter_add_(0, dst_all, torch.ones_like(dst_all, dtype=torch.float))
        deg[deg == 0] = 1

        agg = torch.zeros_like(x)
        agg.index_add_(0, dst_all, x[src_all])
        agg = agg / deg.unsqueeze(1)

        return self.linear(agg)

### Directed GCN Layer


In [6]:
class DirectedGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_in = nn.Linear(in_dim, out_dim)
        self.lin_out = nn.Linear(in_dim, out_dim)

    def forward(self, x, edge_index, num_nodes):
        src, dst = edge_index

        # Incoming aggregation
        deg_in = torch.zeros(num_nodes, device=x.device)
        deg_in.scatter_add_(0, dst, torch.ones_like(dst, dtype=torch.float))
        deg_in[deg_in == 0] = 1

        agg_in = torch.zeros_like(x)
        agg_in.index_add_(0, dst, x[src])
        agg_in = agg_in / deg_in.unsqueeze(1)

        # Outgoing aggregation
        deg_out = torch.zeros(num_nodes, device=x.device)
        deg_out.scatter_add_(0, src, torch.ones_like(src, dtype=torch.float))
        deg_out[deg_out == 0] = 1

        agg_out = torch.zeros_like(x)
        agg_out.index_add_(0, src, x[dst])
        agg_out = agg_out / deg_out.unsqueeze(1)

        return self.lin_in(agg_in) + self.lin_out(agg_out)

## 2. Model Definitions


In [8]:
class BasicGCN(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.layer1 = BasicGCNLayer(input_dim, hidden_dim)
        self.layer2 = BasicGCNLayer(hidden_dim, 2)

    def forward(self, data):
        x = F.relu(self.layer1(data.x, data.edge_index, data.num_nodes))
        return self.layer2(x, data.edge_index, data.num_nodes)


class DirectedGCN(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.layer1 = DirectedGCNLayer(input_dim, hidden_dim)
        self.layer2 = DirectedGCNLayer(hidden_dim, 2)

    def forward(self, data):
        x = F.relu(self.layer1(data.x, data.edge_index, data.num_nodes))
        return self.layer2(x, data.edge_index, data.num_nodes)


class IncomingOnlyLayer(DirectedGCNLayer):
    def forward(self, x, edge_index, num_nodes):
        src, dst = edge_index

        deg_in = torch.zeros(num_nodes, device=x.device)
        deg_in.scatter_add_(0, dst, torch.ones_like(dst, dtype=torch.float))
        deg_in[deg_in == 0] = 1

        agg_in = torch.zeros_like(x)
        agg_in.index_add_(0, dst, x[src])
        agg_in = agg_in / deg_in.unsqueeze(1)

        return self.lin_in(agg_in)


class OutgoingOnlyLayer(DirectedGCNLayer):
    def forward(self, x, edge_index, num_nodes):
        src, dst = edge_index

        deg_out = torch.zeros(num_nodes, device=x.device)
        deg_out.scatter_add_(0, src, torch.ones_like(src, dtype=torch.float))
        deg_out[deg_out == 0] = 1

        agg_out = torch.zeros_like(x)
        agg_out.index_add_(0, src, x[dst])
        agg_out = agg_out / deg_out.unsqueeze(1)

        return self.lin_out(agg_out)


class IncomingGCN(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.layer1 = IncomingOnlyLayer(input_dim, hidden_dim)
        self.layer2 = IncomingOnlyLayer(hidden_dim, 2)

    def forward(self, data):
        x = F.relu(self.layer1(data.x, data.edge_index, data.num_nodes))
        return self.layer2(x, data.edge_index, data.num_nodes)


class OutgoingGCN(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.layer1 = OutgoingOnlyLayer(input_dim, hidden_dim)
        self.layer2 = OutgoingOnlyLayer(hidden_dim, 2)

    def forward(self, data):
        x = F.relu(self.layer1(data.x, data.edge_index, data.num_nodes))
        return self.layer2(x, data.edge_index, data.num_nodes)

## 3. Training Utility

In [9]:
def train_model(model, data, epochs=50, lr=0.001):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    data = data.to(device)
    model = model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    start = time.time()

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        out = model(data)
        loss = criterion(out[data.train_mask], data.y[data.train_mask])

        loss.backward()
        optimizer.step()

    runtime = time.time() - start

    model.eval()
    with torch.no_grad():
        logits = model(data)
        preds = logits[data.test_mask].argmax(dim=1)
        probs = F.softmax(logits[data.test_mask], dim=1)[:, 1]

    results = evaluate_predictions(
        data.y[data.test_mask].cpu(),
        preds.cpu(),
        probs.cpu()
    )

    results["Runtime"] = runtime
    return results

## 4. Single Run Experimental Comparison

In [10]:
results = []

models_to_test = [
    (BasicGCN, "Basic GCN"),
    (DirectedGCN, "Directed GCN (Full)"),
    (IncomingGCN, "Incoming Only"),
    (OutgoingGCN, "Outgoing Only"),
]

for model_class, name in models_to_test:
    print(f"Training {name}...")
    model = model_class(data.x.shape[1])
    res = train_model(model, data)
    results.append({"Model": name, **res})

comparison_df = pd.DataFrame(results)
comparison_df

Training Basic GCN...
Training Directed GCN (Full)...
Training Incoming Only...
Training Outgoing Only...


,Model,F1,Precision,Recall,AUC,Runtime
0,Basic GCN,0.264538,0.166547,0.642659,0.762879,29.665856
1,Directed GCN (Full),0.243568,0.148684,0.673130,0.768349,38.667644
2,Incoming Only,0.150324,0.163067,0.139428,0.595668,19.358211
3,Outgoing Only,0.077166,0.062217,0.101570,0.608509,19.359927


In [11]:
datasets = {}

for dataset_name in ["elliptic", "paysim"]:
    datasets[dataset_name] = load_graph(dataset_name)

Loaded elliptic
Data(x=[203769, 165], edge_index=[2, 234355], y=[203769], timesteps=[203769], num_nodes=203769, train_mask=[203769], test_mask=[203769], val_mask=[203769])
Loaded paysim
Data(x=[592288, 8], edge_index=[2, 325933], y=[592288], num_nodes=592288, train_mask=[592288], val_mask=[592288], test_mask=[592288])


In [12]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

## 5. Multi-Seed Training Function

In [13]:
def train_once(model_class, data, seed=42, hidden_dim=64, epochs=50, lr=0.001):
    
    set_seed(seed)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    data = data.to(device)
    
    model = model_class(data.x.shape[1], hidden_dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    start_time = time.time()
    
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        
        out = model(data)
        loss = criterion(out[data.train_mask], data.y[data.train_mask])
        
        loss.backward()
        optimizer.step()
    
    runtime = time.time() - start_time
    
    model.eval()
    with torch.no_grad():
        logits = model(data)
        preds = logits[data.test_mask].argmax(dim=1)
        probs = F.softmax(logits[data.test_mask], dim=1)[:, 1]
    
    metrics = evaluate_predictions(
        data.y[data.test_mask].cpu(),
        preds.cpu(),
        probs.cpu()
    )
    
    metrics["Runtime"] = runtime
    
    return metrics

In [14]:
def run_experiment(model_class, model_name, dataset_name, data, seeds=[1,2,3,4,5], hidden_dim=64):
    
    all_results = []
    
    for seed in seeds:
        metrics = train_once(
            model_class,
            data,
            seed=seed,
            hidden_dim=hidden_dim
        )
        
        metrics["Dataset"] = dataset_name
        metrics["Model"] = model_name
        metrics["Hidden_Dim"] = hidden_dim
        metrics["Seed"] = seed
        
        all_results.append(metrics)
    
    df = pd.DataFrame(all_results)
    
    summary = {
        "Dataset": dataset_name,
        "Model": model_name,
        "Hidden_Dim": hidden_dim,
        "F1_mean": df["F1"].mean(),
        "F1_std": df["F1"].std(),
        "F1_ci95": 1.96 * df["F1"].std() / np.sqrt(len(seeds)),
        "AUC_mean": df["AUC"].mean(),
        "AUC_std": df["AUC"].std(),
        "AUC_ci95": 1.96 * df["AUC"].std() / np.sqrt(len(seeds)),
        "Runtime_mean": df["Runtime"].mean(),
        "Runtime_std": df["Runtime"].std()
    }
    
    return summary, df

In [15]:
models_to_test = [
    (DirectedGCN, "Directed GCN"),
    (IncomingGCN, "Incoming Only"),
    (OutgoingGCN, "Outgoing Only"),
]

In [16]:
experiment_summaries = []
detailed_results = {}

for dataset_name, data in datasets.items():
    
    print(f"\n Running experiments on {dataset_name.upper()} ")
    
    for model_class, model_name in models_to_test:
        
        summary, detailed_df = run_experiment(
            model_class,
            model_name,
            dataset_name,
            data,
            seeds=[1,2,3,4,5],
            hidden_dim=64
        )
        
        experiment_summaries.append(summary)
        
        detailed_results[f"{dataset_name}_{model_name}"] = detailed_df
        
        print(f"Completed: {model_name} on {dataset_name}")


 Running experiments on ELLIPTIC 
Completed: Directed GCN on elliptic
Completed: Incoming Only on elliptic
Completed: Outgoing Only on elliptic

 Running experiments on PAYSIM 
Completed: Directed GCN on paysim


Completed: Incoming Only on paysim


Completed: Outgoing Only on paysim


## 6. Final Results Aggregation

In [17]:
final_results_df = pd.DataFrame(experiment_summaries)

final_results_df = final_results_df.sort_values(
    by=["Dataset", "F1_mean"],
    ascending=[True, False]
).reset_index(drop=True)

final_results_df

,Dataset,Model,Hidden_Dim,F1_mean,F1_std,F1_ci95,AUC_mean,AUC_std,AUC_ci95,Runtime_mean,Runtime_std
0,elliptic,Directed GCN,64,0.274311,0.027047,0.023708,0.787408,0.014387,0.012611,45.310211,10.823829
1,elliptic,Incoming Only,64,0.153604,0.025942,0.022739,0.595833,0.012381,0.010853,24.209705,4.273703
2,elliptic,Outgoing Only,64,0.111340,0.055820,0.048928,0.653130,0.013435,0.011777,21.364204,3.157161
3,paysim,Directed GCN,64,0.059251,0.033584,0.029438,0.726187,0.159257,0.139594,33.463171,0.315168
4,paysim,Incoming Only,64,0.009817,0.021952,0.019241,0.727522,0.000000,0.000000,17.127782,0.067184
5,paysim,Outgoing Only,64,0.000000,0.000000,0.000000,0.272516,0.000000,0.000000,17.121400,0.084353


## 7. Model Saving

In [24]:
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

save_path = MODEL_DIR / "directed_gcn_v1.pt"

torch.save({
    "model_state_dict": model.state_dict(),
    "input_dim": data.x.shape[1],
    "hidden_dim": 64,
    "dataset": dataset_name,
    "epochs": 50
}, save_path)

print(f"Model saved properly at: {save_path}")

Model saved properly at: ..\models\directed_gcn_v1.pt
